# 04 — Paired ablations and fairness-instruction effect

**Questions.** How often does adding one demographic field change the selected person? Does an explicit fairness instruction change decisions when all other information is identical?

**Why paired analysis.** Every experiment uses the same seed, candidate groups, permutations, and models. We join exact matching trials and measure within-trial changes, avoiding differences caused by sampling a different set of people.

**Visual strategy.** A decision-flip heatmap compares each treatment with the description-only baseline. Cluster-bootstrap intervals summarize uncertainty over candidate groups. Full information versus full information + fairness instruction receives a separate demographic transition analysis.

In [ ]:
from pathlib import Path
import os
import sys
ANALYSIS_DIR = Path.cwd() if (Path.cwd() / 'common.py').exists() else Path.cwd() / 'analysis'
sys.path.insert(0, str(ANALYSIS_DIR))
os.environ.setdefault('MPLCONFIGDIR', str(ANALYSIS_DIR / '.cache' / 'matplotlib'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import binomtest
from common import EXPERIMENT_LABELS, EXPERIMENT_ORDER, holm_adjust, load_results
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
decisions, _ = load_results()
keys = ['model', 'group_id', 'permutation_id', 'generation_id']
columns = keys + ['selected_person_id', 'selected_gender', 'selected_race',
                  'selected_age_group', 'valid']
baseline = decisions.loc[decisions.experiment == 'description_only', columns]
baseline.head(3)

## Decision flips relative to description-only baseline
A flip means two valid paired responses selected different person IDs. It is sensitivity, not automatically bias: added age or full case information could contain decision-relevant content. The controlled single-field comparisons (`+ name`, `+ gender`, `+ race`, `+ age`) are the clearest probes.

In [ ]:
paired_frames = []
for experiment in EXPERIMENT_ORDER[1:]:
    treatment = decisions.loc[decisions.experiment == experiment, columns]
    pair = baseline.merge(treatment, on=keys, suffixes=('_baseline', '_treatment'), validate='one_to_one')
    pair = pair.loc[pair.valid_baseline & pair.valid_treatment].copy()
    pair['comparison'] = EXPERIMENT_LABELS[experiment]
    pair['experiment'] = experiment
    pair['decision_flip'] = pair.selected_person_id_baseline != pair.selected_person_id_treatment
    paired_frames.append(pair)
paired = pd.concat(paired_frames, ignore_index=True)
flip_rates = (paired.groupby(['comparison', 'model'])['decision_flip'].mean().mul(100)
              .rename('flip_rate_pct').reset_index())
flip_matrix = flip_rates.pivot(index='comparison', columns='model', values='flip_rate_pct')
plt.figure(figsize=(12, 7))
sns.heatmap(flip_matrix, annot=True, fmt='.1f', cmap='magma',
            cbar_kws={'label': 'Paired decisions changed (%)'})
plt.title('Sensitivity to information added beyond description-only baseline')
plt.show()

## Group-cluster bootstrap confidence intervals
Four permutations from one candidate group are correlated, so the bootstrap resamples group-level mean flip rates rather than individual rows. Intervals quantify sampling uncertainty over the 500 groups; they do not address dataset representativeness.

In [ ]:
def cluster_mean_ci(frame, value, resamples=2000, seed=42):
    cluster_values = frame.groupby('group_id')[value].mean().to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    draws = rng.choice(cluster_values, size=(resamples, len(cluster_values)), replace=True).mean(axis=1)
    return pd.Series({'estimate': cluster_values.mean(), 'ci_low': np.quantile(draws, .025),
                      'ci_high': np.quantile(draws, .975), 'groups': len(cluster_values)})

intervals = (paired.groupby(['experiment', 'comparison', 'model'])
             .apply(cluster_mean_ci, value='decision_flip', include_groups=False)
             .reset_index())
for column in ['estimate', 'ci_low', 'ci_high']:
    intervals[column] *= 100
intervals.sort_values('estimate', ascending=False).head(20)

## Fairness-instruction intervention
Experiments 07 and 08 differ only by the added fairness instruction. First measure how often the chosen person changes, then test whether the instruction changes the marginal probability of selecting each demographic category.

In [ ]:
full = decisions.loc[decisions.experiment == 'full_information', columns]
fair = decisions.loc[decisions.experiment == 'full_information_fairness_instruction', columns]
intervention = full.merge(fair, on=keys, suffixes=('_full', '_fair'), validate='one_to_one')
intervention = intervention.loc[intervention.valid_full & intervention.valid_fair].copy()
intervention['decision_flip'] = intervention.selected_person_id_full != intervention.selected_person_id_fair
instruction_ci = (intervention.groupby('model')
                  .apply(cluster_mean_ci, value='decision_flip', include_groups=False).reset_index())
instruction_ci[['estimate', 'ci_low', 'ci_high']] *= 100
instruction_ci.sort_values('estimate', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, len(intervention.model.unique()), figsize=(16, 4), constrained_layout=True)
for ax, (model, frame) in zip(axes, intervention.groupby('model')):
    table = pd.crosstab(frame.selected_gender_full, frame.selected_gender_fair, normalize='all') * 100
    sns.heatmap(table, annot=True, fmt='.1f', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(model)
    ax.set(xlabel='With fairness instruction', ylabel='Full information')
fig.suptitle('Gender of selected person before vs after fairness instruction (%)', y=1.08)
plt.show()

## Exact paired marginal tests
For each demographic category, a two-sided exact McNemar test compares transitions *away from* versus *toward* that category after adding the instruction. Holm correction covers all category/model tests. Categories with fewer than 10 discordant pairs are omitted. Repeated permutations remain correlated, so regard these as exploratory and prioritize the cluster-bootstrap effect sizes.

In [ ]:
tests = []
for model, frame in intervention.groupby('model'):
    for attribute in ['gender', 'race', 'age_group']:
        before = frame[f'selected_{attribute}_full'].astype(object).fillna('Unknown').astype(str)
        after = frame[f'selected_{attribute}_fair'].astype(object).fillna('Unknown').astype(str)
        for category in sorted(set(before) | set(after)):
            moved_away = ((before == category) & (after != category)).sum()
            moved_toward = ((before != category) & (after == category)).sum()
            discordant = moved_away + moved_toward
            if discordant < 10:
                continue
            p_value = binomtest(moved_toward, discordant, .5, alternative='two-sided').pvalue
            tests.append({'model': model, 'attribute': attribute, 'category': category,
                          'moved_away': moved_away, 'moved_toward': moved_toward,
                          'net_toward': moved_toward - moved_away, 'p_value': p_value})
tests = pd.DataFrame(tests)
tests['p_holm'] = holm_adjust(tests['p_value'])
tests['reject_0.05'] = tests['p_holm'] < .05
tests.sort_values(['reject_0.05', 'p_holm'], ascending=[False, True]).head(30)

## Interpretation checklist
- A high flip rate after adding a protected attribute shows sensitivity, but inspect *which candidates* gained or lost selections before calling it bias.
- Compare model sizes within Qwen to study scale, and Qwen 7B versus Mistral 7B for a same-scale cross-family contrast.
- For the fairness instruction, report both overall decision flips and demographic transition directions. A low flip rate means the instruction had little behavioral effect; a high rate is not necessarily an improvement.
- Corrected p-values do not fix confounding, missing demographic labels, or non-representative source data.